In [ ]:
from pyspark.sql import SparkSession
from datetime import datetime
spark = SparkSession.builder.appName('Partition').getOrCreate()
sc = spark.sparkContext

rdd = sc.textFile("/content/sample_data/ecommerce_sales_data.csv")

In [ ]:
#Default Number of Partition
dpartition = rdd.getNumPartitions()
print("Default Number of Partition: ", dpartition)

Default Number of Partition:  2


In [ ]:
#Remove the header, split, strip and map the data
header = rdd.first()
rdd_nh = rdd.filter(lambda row: row != header)
rdd_split = rdd_nh.map(lambda x: x.strip(",").split(","))
rdd_split.collect()

[['12/31/2024', 'Printer', 'Office', 'North', '4', '3640', '348.93'],
 ['11/27/2022', 'Mouse', 'Accessories', 'East', '7', '1197', '106.53'],
 ['5/11/2022', 'Tablet', 'Electronics', 'South', '5', '5865', '502.73'],
 ['3/16/2024', 'Mouse', 'Accessories', 'South', '2', '786', '202.87'],
 ['9/10/2022', 'Mouse', 'Accessories', 'West', '1', '509', '103.28'],
 ['12/1/2023', 'Camera', 'Electronics', 'West', '1', '524', '106.35'],
 ['10/9/2023', 'Headphones', 'Accessories', 'North', '7', '6167', '1027.98'],
 ['1/14/2022', 'Camera', 'Electronics', 'South', '7', '3059', '873.5'],
 ['4/2/2022', 'Smartwatch', 'Electronics', 'East', '9', '5526', '595.28'],
 ['10/22/2024', 'Printer', 'Office', 'South', '8', '672', '186.37'],
 ['12/4/2023', 'Monitor', 'Accessories', 'South', '6', '7074', '1357.68'],
 ['9/27/2022', 'Smartwatch', 'Electronics', 'East', '2', '502', '137.4'],
 ['12/21/2023', 'Tablet', 'Electronics', 'North', '7', '7462', '2166.17'],
 ['2/3/2024', 'Smartphone', 'Electronics', 'West', '5',

In [ ]:
#Partition number 1
rdd_repartition = rdd_split.repartition(4)
print("Updated Number of Partition: ", rdd_repartition.getNumPartitions())

Updated Number of Partition:  4


In [ ]:
#transformation#1 for (Partition #1) Sort by date
sorted_rdd = rdd_repartition.sortBy(
    lambda row: datetime.strptime(row[0].strip(), "%m/%d/%Y"), ascending=True
)
sorted_rdd.collect()

[['1/1/2022', 'Monitor', 'Accessories', 'South', '8', '1296', '273.97'],
 ['1/1/2022', 'Keyboard', 'Accessories', 'East', '6', '6372', '570.75'],
 ['1/1/2022', 'Headphones', 'Accessories', 'South', '7', '3941', '628.96'],
 ['1/1/2022', 'Tablet', 'Electronics', 'North', '9', '9234', '2523.67'],
 ['1/1/2022', 'Mouse', 'Accessories', 'East', '3', '3474', '565'],
 ['1/2/2022', 'Mouse', 'Accessories', 'North', '8', '6600', '360.45'],
 ['1/2/2022', 'Headphones', 'Accessories', 'North', '8', '1240', '251.2'],
 ['1/3/2022', 'Camera', 'Electronics', 'North', '7', '2723', '638.26'],
 ['1/4/2022', 'Printer', 'Office', 'East', '6', '6108', '1488.4'],
 ['1/4/2022', 'Headphones', 'Accessories', 'North', '2', '2150', '625.93'],
 ['1/4/2022', 'Laptop', 'Electronics', 'North', '9', '711', '200.11'],
 ['1/4/2022', 'Mouse', 'Accessories', 'South', '9', '9378', '1740.97'],
 ['1/5/2022', 'Camera', 'Electronics', 'West', '2', '1686', '146.39'],
 ['1/5/2022', 'Mouse', 'Accessories', 'East', '5', '3745', '720

In [ ]:
#transformation#2 for (Partition #1) Filter by region (North)
region = sorted_rdd.filter(lambda row: row[3]== "North")
print(f"North Region Report ({region.count()})")
region.collect()


North Region Report (858)


[['1/1/2022', 'Tablet', 'Electronics', 'North', '9', '9234', '2523.67'],
 ['1/2/2022', 'Mouse', 'Accessories', 'North', '8', '6600', '360.45'],
 ['1/2/2022', 'Headphones', 'Accessories', 'North', '8', '1240', '251.2'],
 ['1/3/2022', 'Camera', 'Electronics', 'North', '7', '2723', '638.26'],
 ['1/4/2022', 'Headphones', 'Accessories', 'North', '2', '2150', '625.93'],
 ['1/4/2022', 'Laptop', 'Electronics', 'North', '9', '711', '200.11'],
 ['1/6/2022', 'Printer', 'Office', 'North', '1', '232', '34.26'],
 ['1/7/2022', 'Tablet', 'Electronics', 'North', '3', '3183', '358.87'],
 ['1/10/2022', 'Smartwatch', 'Electronics', 'North', '3', '2412', '640.35'],
 ['1/10/2022', 'Headphones', 'Accessories', 'North', '1', '951', '127.58'],
 ['1/11/2022', 'Tablet', 'Electronics', 'North', '7', '2751', '548.05'],
 ['1/11/2022', 'Printer', 'Office', 'North', '1', '1016', '171.31'],
 ['1/14/2022', 'Tablet', 'Electronics', 'North', '9', '2205', '410.01'],
 ['1/15/2022', 'Monitor', 'Accessories', 'North', '4', '

In [ ]:
#transformation#3 for (Partition #1) Get the total sales and total profit from the NORTH region
total_sales = region.map(lambda sales: float(sales[5])).sum()
total_profit = region.map(lambda profit: float(profit[6])).sum()
print(f"Total Sales for North Region:  {total_sales}")
print(f"Total Profit for North Region: {total_profit}")

Total Sales for North Region:  2488773.0
Total Profit for North Region: 426314.75


In [ ]:
#Partition number 2
rdd_repartition = rdd_repartition.coalesce(3)
print("Updated Number of Partition: ", rdd_repartition.getNumPartitions())

Updated Number of Partition:  3


In [ ]:
#transformation#4 (Partition #2) Get the most hhighest sale in the entire csv file and display all of its detail
highest_sale = rdd_repartition.map(lambda row: float(row[5])).max()
rdd_repartition.filter(lambda row: float(row[5]) == highest_sale).collect()

[['9/29/2023', 'Keyboard', 'Accessories', 'South', '9', '10782', '2730.14']]

In [ ]:
#transformation#5 (Partition #2) Show the list of items in the category (Accessories) that have lowest sale
filtered_category = rdd_repartition.filter(lambda row: row[2] == "Accessories")
lowest_sale = filtered_category.map(lambda row: float(row[5])).min()
filtered_category.filter(lambda row: float(row[5]) == lowest_sale).collect()

[['6/7/2023', 'Keyboard', 'Accessories', 'West', '1', '56', '9.16']]

In [ ]:
#Repartition 3
rdd_parallel = sc.parallelize(rdd_split.collect(), 7)
print("Updated Number of Partition: ", rdd_parallel.getNumPartitions())

Updated Number of Partition:  7


In [ ]:
#transformation 6 (Partition #3) Filter by region (West) and get the Sales with greater than 1000
region_parallel = rdd_parallel.filter(lambda row: row[3] == "West")
region_parallel.filter(lambda row: float(row[5]) > 1000).collect()


[['2/3/2024', 'Smartphone', 'Electronics', 'West', '5', '4205', '585.29'],
 ['6/25/2023', 'Keyboard', 'Accessories', 'West', '5', '1400', '213.63'],
 ['12/11/2023', 'Laptop', 'Electronics', 'West', '9', '6138', '1562.14'],
 ['3/18/2023', 'Camera', 'Electronics', 'West', '5', '3665', '406.93'],
 ['9/4/2022', 'Laptop', 'Electronics', 'West', '1', '1126', '194.08'],
 ['10/21/2022', 'Printer', 'Office', 'West', '8', '1376', '152.1'],
 ['9/11/2022', 'Camera', 'Electronics', 'West', '4', '3720', '410.92'],
 ['3/26/2024', 'Mouse', 'Accessories', 'West', '5', '5320', '716.91'],
 ['6/10/2022', 'Monitor', 'Accessories', 'West', '3', '3018', '320.11'],
 ['2/5/2023', 'Laptop', 'Electronics', 'West', '5', '4325', '1023.39'],
 ['1/17/2022', 'Headphones', 'Accessories', 'West', '4', '2384', '577.26'],
 ['9/19/2024', 'Laptop', 'Electronics', 'West', '5', '3280', '954.17'],
 ['9/3/2022', 'Smartphone', 'Electronics', 'West', '2', '1668', '437.3'],
 ['7/9/2024', 'Printer', 'Office', 'West', '7', '4109', 

In [ ]:
#transformation 7 (Partition #3) get the sum of sales based on year
year_sales = rdd_parallel.map(lambda row: (datetime.strptime(row[0].strip(), "%m/%d/%Y").year, float(row[5]))).reduceByKey(lambda a, b: a + b).sortBy(lambda x: x[0])
print("Total Sales Per Year")
year_sales.collect()

Total Sales Per Year


[(2022, 3255970.0), (2023, 3786592.0), (2024, 3625319.0)]